# Transformers for Audio Tasks

## 1. Audio Classification and Event Detection

Categorizing audio data into predefined classes (e.g., identifying a "dog bark," "glass breaking," or speaker intent/emotion).

Representative Models:

- **Audio Spectrogram Transformer (AST)** for environmental sounds.
- fine-tuned **Wav2Vec2** for speaker intent or language identification.

In [ ]:
from datasets import load_dataset
from datasets import Audio

minds = load_dataset("PolyAI/minds14", name="en-AU", split="train")
minds = minds.cast_column("audio", Audio(sampling_rate=16_000))


To classify an audio recording into a set of classes, we can use the `audio-classification` pipeline from 🤗 Transformers. In our case, we need a model that's been fine-tuned for intent classification, and specifically on the MINDS-14 dataset. Luckily for us, the Hub has a model that does just that! Let's load it by using the `pipeline()` function:

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "audio-classification",
    model="anton-l/xtreme_s_xlsr_300m_minds14",
)

This pipeline expects the audio data as a NumPy array. All the preprocessing of the raw audio data will be conveniently handled for us by the pipeline. Let's pick an example to try it out:

In [ ]:
example = minds[0]

If you recall the structure of the dataset, the raw audio data is stored in a NumPy array under `["audio"]["array"]`, let's pass it straight to the `classifier`:

In [ ]:
classifier(example["audio"]["array"])

**Output:**
```out
[
    {"score": 0.9631525278091431, "label": "pay_bill"},
    {"score": 0.02819698303937912, "label": "freeze"},
    {"score": 0.0032787492964416742, "label": "card_issues"},
    {"score": 0.0019414445850998163, "label": "abroad"},
    {"score": 0.0008378693601116538, "label": "high_value_payment"},
]
```

The model is very confident that the caller intended to learn about paying their bill. Let's see what the actual label for this example is:

In [ ]:
id2label = minds.features["intent_class"].int2str
id2label(example["intent_class"])

**Output:**
```out
"pay_bill"
```

Hooray! The predicted label was correct! Here we were lucky to find a model that can classify the exact labels that we need. A lot of the times, when dealing with a classification task, a pre-trained model's set of classes is not exactly the same as the classes you need the model to distinguish. In this case, you can fine-tune a pre-trained model to "calibrate" it to your exact set of class labels. We'll learn how to do this in the upcoming units. Now, let's take a look at another very common task in speech processing, _automatic speech recognition_.

## 2. Automatic Speech Recognition (ASR)

Converting spoken language into written text for transcription, voice commands, and accessibility.

Representative Models:

- **Whisper** (OpenAI) for robust multilingual transcription
- **SeamlessM4T** (Meta) for direct speech-to-text translation.


![](../assets/asr.png)

### Applications

- **Transcription of:**
  - **phone calls** for customer service and sales
  - **meetings** for note-taking and documentation
  - **lectures** for students and researchers
  - **podcasts** for SEO and accessibility
  - **interviews** for journalists and researchers
- **Dictation:** for people to speak their notes directly into text documents.
- **Voice Assistants:** AI-powered devices can answer questions, set reminders, and control smart homes (digitally connected).

We can fit a small model like [`openai/whisper-large-v3-turbo`](https://huggingface.co/openai/whisper-large-v3-turbo) (800 million parameters) and use it according to the model card [usage section](https://huggingface.co/openai/whisper-large-v3-turbo#usage).

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe_asr = pipeline(
    task="automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch_dtype,
    device=device,
)

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [ ]:
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
sample = dataset[0]["audio"]

### ASR Parameters

In [ ]:
transcription = pipe_asr(
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac",
    return_timestamps="word", # Returns precise start/end times for every word
    chunk_length_s=30,        # Splits long audio into 30-second segments for processing
    stride_length_s=5         # Adds 5 seconds of overlap between segments to prevent cutting words
)

In [ ]:
print(transcription)

{
    'text': " Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel. Nor is Mr. 
Quilter's manner less interesting than his matter. He tells us that at this festive season of the year, with 
Christmas and roast beef looming before us, similes drawn from eating and its results occur most readily to the 
mind. He has grave doubts whether Sir Frederick Layton's work is really Greek after all, and can discover in it but
little of rocky Ithaca. Linnell's pictures are a sort of Up Guards and Adam paintings, and Mason's exquisite idles 
are as national as a jingo poem. Mr. Birkett Foster's landscapes smile at one much in the same way that Mr. Carker 
used to flash his teeth, and Mr. John Collier gives his sitter a cheerful slap on the back before he says like a 
shampooer in a Turkish bath next man",
    'chunks': [
        {
            'timestamp': (0.0, 5.3),
            'text': ' Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.'
        },
        {'timestamp': (6.36, 10.08), 'text': " Nor is Mr. Quilter's manner less interesting than his matter."},
        {
            'timestamp': (11.04, 17.62),
            'text': ' He tells us that at this festive season of the year, with Christmas and roast beef looming 
before us,'
        },
        {
            'timestamp': (18.46, 22.6),
            'text': ' similes drawn from eating and its results occur most readily to the mind.'
        },
        {
            'timestamp': (23.1, 28.68),
            'text': " He has grave doubts whether Sir Frederick Layton's work is really Greek after all,"
        },
        {'timestamp': (29.08, 28.68), 'text': ''},
        {'timestamp': (32.48, 33.66), 'text': ' and can discover in it but little of rocky Ithaca.'},
        {'timestamp': (37.84, 38.24), 'text': " Linnell's pictures are a sort of Up Guards and Adam paintings,"},
        {'timestamp': (42.86, 44.5), 'text': " and Mason's exquisite idles are as national as a jingo poem."},
        {
            'timestamp': (47.64, 51.56),
            'text': " Mr. Birkett Foster's landscapes smile at one much in the same way that Mr. Carker used to 
flash his teeth,"
        },
        {
            'timestamp': (52.4, 57.24),
            'text': ' and Mr. John Collier gives his sitter a cheerful slap on the back'
        },
        {'timestamp': (57.24, 60.74), 'text': ' before he says like a shampooer in a Turkish bath'},
        {'timestamp': (60.74, 61.92), 'text': ' next man'}
    ]
}

### Fine-tuned ASR

> Example: [NAMAA-Space/EgypTalk-ASR-v2](https://huggingface.co/NAMAA-Space/EgypTalk-ASR-v2) trained on over 200 hours of high-quality, manually curated audio data collected and prepared by the NAMAA team. It is built upon NVIDIA’s FastConformer Hybrid Large architecture and fine-tuned for Egyptian Arabic, enabling highly accurate transcription in casual, formal, and mixed dialect settings.

## 3. Text-to-Speech (TTS)

Also known as: Voice Synthesis; Generating natural-sounding human speech from text, including voice cloning and emotional modulation for agents, audiobooks, and dubbing.

Representative Models:

- **Qwen3-TTS** or **CosyVoice2** for ultra-low latency, mobile-optimized streaming
- **VALL-E** (implementations) for zero-shot voice cloning.

> Example: [NAMAA-Saudi-TTS](https://huggingface.co/spaces/omarelshehy/NAMAA-Saudi-Voice) refined to generate natural Saudi dialect speech, targeting everyday conversational usage rather than Modern Standard Arabic (MSA).

### Applications

1. **Customer Support (IVR)**: Operating automated phone menus and voice-based customer service bots to resolve user issues.
2. **Virtual assistants**: Once they have used a classification model to catch the *wake word*, and used ASR model to process your request, they can use a TTS model to respond to your inquiry.
3. **Navigation Systems**: Delivering turn-by-turn GPS directions and real-time traffic updates while driving
4. **Listen rather than read**: Converting novels, news articles, and blogs into audio formats for hands-free, on-the-go consumption.
      - *visually impaired users*
      - *reading difficulties*
      - *dyslexia*

In [ ]:
from transformers import pipeline

tts = pipeline(
    "text-to-speech",
    model="omarelshehy/NAMAA-Saudi-Voice"
)

text = "أهلاً وسهلاً بكم في هذا الدرس التعليمي. نتمنى لكم تجربة ممتعة وثرية."

output = tts(text)

# Save the generated speech to a file
with open("namaa_saudi_tts_output.wav", "wb") as f:
    f.write(output["audio"])

print("TTS audio has been generated and saved as 'namaa_saudi_tts_output.wav'.")

## 4. Voice Activity Detection (VAD)

Tells you where speech begins and ends in an audio.

![](../assets/vad.png)

See: [`voice-activity-detection`](https://huggingface.co/models?pipeline_tag=voice-activity-detection) on the Hub.

## 5. Speaker Diarization

Determining "who spoke when" by segmenting audio recordings according to distinct speaker identities—crucial for meetings, interviews, and call center analytics.

![](../assets/speaker_diarization.png)

Representative Models:

- **pyannote-audio** pipeline (pretrained on VoxCeleb and AMI datasets) is widely used for robust, plug-and-play diarization.
- **UIS-RNN** for online and streaming scenarios with incremental speaker assignment.

See: [`speaker-diarization`](https://huggingface.co/models?other=speaker-diarization) on the Hub.

## 6. Music-source Separation

Isolating specific audio tracks from a mixture or cleaning up noisy audio (e.g., separating vocals from instruments, removing background static).

![](../assets/music_source_separation.png)

Representative Models:

- **HTDemucs** for music stem separation.
- **SepFormer** for isolating clean speech from complex acoustic environments.

See: [`music-source-separation`](https://huggingface.co/models?other=music-source-separation&sort=trending) on the Hub.